# Brazilian Tourism Data Pipeline (1989-2024)

**Source:** [Ministério do Turismo - Dados Abertos](https://dados.gov.br/dados/conjuntos-dados/estimativas-de-chegadas-de-turistas-internacionais-ao-brasil)

## Project Overview
This project demonstrates the end-to-end process of handling real-world historical data. 
It focuses on building a robust ETL (Extract, Transform, Load) pipeline to consolidate 
over 30 years of Brazilian tourism records, addressing challenges like schema drift 
and data inconsistency.

---

## Table of Contents
0. [Setup](#setup)
1. [Schema Discovery](#schema-discovery)
2. [Mapping Strategy](#normalization)
3. [Data Pipeline](#pipeline)<br>
&nbsp;3.1. [Transformation Logic](#logic)<br>
&nbsp;3.2. [Unit Test & Quality Audit](#qualityaudit)<br>
&nbsp;3.3. [Data Aggregation](#aggregation)
4. [Quality Check](#consolidation)<br>
&nbsp;4.1. [Checking NaN Values](#nancheck)<br>
5. [Database Export](#database)

<a id="setup"></a>
## 0. Environment Setup & Configuration
Initializes necessary libraries and defines global settings for data handling and visualization.

In [1]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

#Global Configuration
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

<a id="schema-discovery"></a>
## 1. Schema Discovery
The first step of our pipeline is to "scan" the raw files. Since this dataset spans 
decades, column names often change. We identify these variations to build a 
mapping strategy without loading the entire dataset into memory.


In [2]:
def get_data_inventory(directory_path: str, separator: str = ';'):
    """
    Scans all CSV files to check if their structure match.
    Returns a list of dictionaries with the different structure among all files.
    """
    path = Path(directory_path)
    files = sorted(list(path.glob("chegadas_*.csv")))
    
    inventory = []
    
    for f in files:
        # Read only the header for efficiency
        header_df = pd.read_csv(f, sep=separator, nrows=0, encoding='latin1')
        cols = header_df.columns.tolist()
        
        inventory.append({
            "filename": f.name,
            "col_count": len(cols),
            "columns": cols
        })
        
    return inventory

In [3]:
# Path to raw data
raw_folder = "../data/raw/"

# Execute inventory scan
files_inventory = get_data_inventory(raw_folder)
inventory_df = pd.DataFrame(files_inventory)

# Filter to show only the unique schemas found
pd.set_option('display.max_colwidth', None)
inventory_df['cols_str'] = inventory_df['columns'].astype(str)

unique_schemas = (
    inventory_df
    .drop_duplicates(subset=['col_count', 'cols_str'])
    .drop(columns=['cols_str'])
)

print(f"{len(unique_schemas)} different types of file structures have been found.\n")
unique_schemas

3 different types of file structures have been found.



,filename,col_count,columns
0,chegadas_1989.csv,12,"[Continente, Ordem continente, País, Ordem país, UF, Ordem UF, Via de acesso, Ordem via de acesso, ano, Mês, Ordem mês, Chegadas]"
11,chegadas_2000.csv,12,"[Continente, Ordem continente, País, Ordem país, UF, Ordem UF, Via de acesso, Ordem via de acesso, Ano, Mês, Ordem mês, Chegadas]"
27,chegadas_2016.csv,12,"[Continente, cod continente, País, cod pais, UF, cod uf, Via, cod via, ano, Mês, cod mes, Chegadas]"


### Analysis of Findings
All files consistently contain **12 columns**, which simplifies the consolidation. However, the inventory reveals three distinct patterns:

* **Pattern 1 & 2:** The primary difference is simple capitalization (e.g., `Ano` vs. `ano`).
* **Pattern 3:** Column names are significantly shortened, and the capitalization remains inconsistent.

These findings confirm that while the "shape" of the data is stable, a **mapping layer** is essential to ensure that columns from different years align perfectly during the merge.

<a id="normalization"></a>
## 2. Mapping Strategy

The next step is to standardize our dataset. We will define our "Source of Truth" by choosing official column names and ensuring every year follows the same schema. 

Additionally, we will eliminate redundant data (like auxiliary codes) identified during the initial analysis of the 1989 file, keeping only what is essential for analysis.

In [4]:
def standardize():
    """
    Returns the mapping dictionary and the list of columns to keep.
    This unifies naming patterns and removes redundant data.
    """
    column_mapping = {
        'Continente': 'continent',
        'País': 'country',
        'UF': 'state',
        'Via de acesso': 'arrival_method', 'Via': 'arrival_method',
        'ano': 'year', 'Ano': 'year',
        'Mês': 'month',
        'Chegadas': 'arrivals'
    }
    target_columns = ['continent', 'country', 'state', 'arrival_method', 'year', 'month', 'arrivals']
    
    return column_mapping, target_columns

In [5]:
# Initialize mapping rules
column_map, target_cols = standardize()

# --- Proof of Concept (Sanity Check) ---
# Testing if the mapping correctly unifies the unique structures from Step 1

def apply_test_mapping(cols_list):
    # Rename and filter in one step for the test
    return [column_map.get(c) for c in cols_list if column_map.get(c) in target_cols]

# Apply the test to unique_schemas table
unique_schemas['standardized_columns'] = unique_schemas['columns'].apply(apply_test_mapping)

# Display result
print("Verification: If all rows in 'standardized_columns' are identical, the mapping is successful.")
unique_schemas[['filename', 'standardized_columns']]

Verification: If all rows in 'standardized_columns' are identical, the mapping is successful.


,filename,standardized_columns
0,chegadas_1989.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
11,chegadas_2000.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
27,chegadas_2016.csv,"[continent, country, state, arrival_method, year, month, arrivals]"


### Strategy Validation
The Proof of Concept was **successful**. Even with different source schemas, the mapping function consistently produced the same set of 7 target columns. 

**Next steps:**
* Transition from header-only analysis to full file processing.
* Loop through the raw directory to consolidate all years into a single master dataset.

<a id="pipeline"></a>
## 3. Data Pipeline
Now we apply our strategy to the actual data. This process is divided into the transformation of individual files and their final consolidation.

<a id="logic"></a>
### 3.1. Transformation Logic

In [6]:
def transform_data(file_path, column_map, target_cols):
    """
    Reads a single file, applies renaming, filters columns, 
    and handles basic data cleaning and string standardization.
    """
    # 1. Extraction (Reading)
    df = pd.read_csv(file_path, sep=';', encoding='latin1')
    
    # 2. Transformation: Column Mapping
    df = df.rename(columns=column_map)
    
    # 3. Transformation: Filtering only target columns
    df = df[df.columns.intersection(target_cols)].copy()
    
    # 4. Data Cleansing: String Standardization
    categorical_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in categorical_cols:
        df[col] = df[col].astype(str).str.strip().str.lower()
    
    # 5. Data Cleansing: Numeric Conversion
    if 'arrivals' in df.columns:
        df['arrivals'] = pd.to_numeric(df['arrivals'], errors='coerce')
    
    # 6. Add metadata for traceability
    df['source_file'] = Path(file_path).name
    
    return df

Decision: We keep arrivals as NaN where parsing fails to avoid injecting bias. These records represent less than 4% of the 1989 data as we analyzed on our previous notebook, and will be handled during the visualization phase according to business requirements.

<a id="qualityaudit"></a>
### 3.2. Unit Test & Quality Audit

Unit Test: 1989 Data
Before running the pipeline through all 35+ files, we perform a **Unit Test** on the 1989 dataset. This year is known for having inconsistent formatting and missing values.

Our goal here is to:
* Identify how many rows contain non-numeric "garbage" in the `arrivals` column.
* Verify if the column mapping correctly captured the historical headers.

In [7]:
# Select file for testing
test_file = "../data/raw/chegadas_1989.csv"

# Execute transformation (Injecting our rules)
df_1989_clean = transform_data(test_file, column_map, target_cols)

# Quality Health Report (Anomaly Detection)
nan_count = df_1989_clean['arrivals'].isna().sum()
total_rows = len(df_1989_clean)

print(f"--- Quality Report: {test_file} ---")
print(f"Total rows processed: {total_rows}")
print(f"Anomalies found (NaN in 'arrivals'): {nan_count}")
print(f"Error Rate: {(nan_count/total_rows)*100:.2f}%")
print("-" * 53)

# Preview transformed data
display(df_1989_clean.head())

--- Quality Report: ../data/raw/chegadas_1989.csv ---
Total rows processed: 17052
Anomalies found (NaN in 'arrivals'): 588
Error Rate: 3.45%
-----------------------------------------------------


,continent,country,state,arrival_method,year,month,arrivals,source_file
0,áfrica,áfrica do sul,amazonas,aérea,1989,janeiro,9.0,chegadas_1989.csv
1,áfrica,angola,amazonas,aérea,1989,janeiro,0.0,chegadas_1989.csv
2,áfrica,nigéria,amazonas,aérea,1989,janeiro,0.0,chegadas_1989.csv
3,áfrica,outros países,amazonas,aérea,1989,janeiro,0.0,chegadas_1989.csv
4,américa central e caribe,costa rica,amazonas,aérea,1989,janeiro,6.0,chegadas_1989.csv


### Unit Test Findings & Data Quality Audit
The test on 1989 data confirms our previous exploratory findings:

Anomaly Consistency: Exactly 588 rows failed numeric conversion (3.45% of the dataset).

Root Cause Identified: Historical records for 'Mato Grosso do Sul' (Fluvial access) contain non-numeric characters that prevent direct calculation.

Pipeline Resilience: By using the coerce strategy, we successfully isolated these anomalies as NaN. This allows the pipeline to continue processing the other 35+ years without crashing, while keeping the "dirty" data visible for later decision-making (e.g., dropping or imputing these values).

<a id='aggregation'></a>
### 3.3. Data Aggregation

With the transformation logic validated by our Unit Test, we now proceed to aggregate the entire historical series (1989-2024).

In [8]:
# List all CSV files in the raw data folder
file_paths = glob.glob("../data/raw/*.csv")

# List to store each cleaned DataFrame
all_dfs = []

print(f"Starting aggregation of {len(file_paths)} files...")

# Loop through files and transform
for file in file_paths:
    try:
        temp_df = transform_data(file, column_map, target_cols)
        all_dfs.append(temp_df)
    except Exception as e:
        print(f"Error processing file {file}: {e}")

# Concatenate everything into one single DataFrame
df_tourism = pd.concat(all_dfs, ignore_index=True)

print("Aggregation complete!")
print(f"Final dataset shape: {df_tourism.shape}")

# Preview the consolidated data
df_tourism.sample(5)

Starting aggregation of 36 files...
Aggregation complete!
Final dataset shape: (953672, 8)


,continent,country,state,arrival_method,year,month,arrivals,source_file
357267,europa,outros países,outras unidades da federação,marítima,2008,outubro,1.0,chegadas_2008.csv
656871,ásia,outros países,mato grosso do sul,fluvial,2018,abril,0.0,chegadas_2018.csv
643486,ásia,outros países,acre,terrestre,2018,novembro,0.0,chegadas_2018.csv
783529,áfrica,nigéria,rio grande do sul,fluvial,2020,março,0.0,chegadas_2020.csv
334458,europa,dinamarca,paraná,marítima,2007,novembro,1.0,chegadas_2007.csv


In [9]:
df_tourism.info()

<class 'pandas.DataFrame'>
RangeIndex: 953672 entries, 0 to 953671
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   continent       953672 non-null  str    
 1   country         953672 non-null  str    
 2   state           953672 non-null  str    
 3   arrival_method  953672 non-null  str    
 4   year            953672 non-null  int64  
 5   month           953672 non-null  str    
 6   arrivals        943820 non-null  float64
 7   source_file     953672 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 58.2 MB


The ETL pipeline has successfully standardized and consolidated the historical series from 1989 to 2024 into a single canonical structure.

**Final Dataset Overview:**
- **Dimensions:** 953,672 rows and 8 columns.
- **Structural Integrity:** 100% of the dimension columns (`continent`, `country`, `state`, `arrival_method`, `year`, `month`) are populated and standardized.
- **Data Quality Audit:** The `arrivals` column contains **~9,852 null values** (representing ~1.03% of the total volume). These gaps are isolated to specific legacy records and have been preserved as `NaN` to maintain statistical honesty for the upcoming analysis.

**Conclusion:**
The dataset is now structurally sound and optimized for the next stages of Data Quality refinement and Database integration.

<a id="consolidation"></a>
## 4. Data Quality & Cleaning
To make sure data is consistent and identify possible mistakes, we will analyze the missing values on arrivals, which is our main column for analysis, but also we will make sure that all values on the other columns are consistent and aligned.
Why? Even if it's only ~1.03% of total values, it can still be a huge amount of data for determined year, so it's important to check how much impact that value really has before making decisions.

<a id='nancheck'></a>
### 4.1. Checking NaN Values.

Before converting `NaN` values to zeros, we perform a root cause analysis.
We verify if the missing data pattern observed in 1989 (Fluvial access in specific states) persists throughout the years.

In [23]:
def audit_missing_values(df):
    """
    Performs a forensic audit on missing values (NaN) in the tourism dataset.
    Identifies problematic years, percentages, and likely causes by grouping
    dimensions (State, Arrival Method, etc.)
    """
    
    # General identification of years with nulls
    null_data = df[df['arrivals'].isna()]
    years_with_nulls = null_data['year'].unique()
    
    if len(years_with_nulls) == 0:
        print("✅ No missing values found in the 'arrivals' column.")
        return None

    report_list = []

    for year in sorted(years_with_nulls):
        yearly_df = df[df['year'] == year]
        total_rows = len(yearly_df)
        nan_rows = yearly_df['arrivals'].isna().sum()
        nan_pct = (nan_rows / total_rows) * 100
        
        # Investigate specific causes:
        # Which states are failing?
        missing_states = yearly_df[yearly_df['arrivals'].isna()]['state'].unique()
        
        # Which methods are failing?
        missing_methods = yearly_df[yearly_df['arrivals'].isna()]['arrival_method'].unique()
        
        # Deep Dive: Is it a specific combination?
        # We group to see if a state lost 100% of its data or just one category
        cause_summary = []
        for state in missing_states:
            state_data = yearly_df[yearly_df['state'] == state]
            state_nan_cnt = state_data['arrivals'].isna().sum()
            state_total = len(state_data)
            
            if state_nan_cnt == state_total:
                cause_summary.append(f"State '{state.upper()}' registered NO data (100% missing)")
            else:
                methods = state_data[state_data['arrivals'].isna()]['arrival_method'].unique()
                cause_summary.append(f"State '{state.upper()}' missing methods: {list(methods)}")

        report_list.append({
            'Year': year,
            'Missing %': round(nan_pct, 2),
            'Total NaN': nan_rows,
            'Affected States': list(missing_states),
            'Likely Causes': " | ".join(cause_summary)
        })

    # Convert to DataFrame for a polished display
    report_df = pd.DataFrame(report_list)
    return report_df

In [14]:
# Isolating 'Santa Catarina' for the year of '1996'
sc_96 = df_tourism[(df_tourism['year'] == 1996) & (df_tourism['state'] == 'santa catarina')]

# 2. calculating NaN % present in 'arrivals'
total_rows = len(sc_96)
nan_count = sc_96['arrivals'].isna().sum()
pct_nan = (nan_count / total_rows) * 100
fluvial_sc = sc_96[sc_96['arrival_method'] == 'fluvial']
fluvial_count = len(fluvial_sc)

print(f"--- Santa Catarina Report (1996) ---")
print(f"Total of entries: {total_rows}")
print(f"Null entries (NaN): {nan_count} ({pct_nan:.2f}%)")
print(f"Fluvial entries: {fluvial_count}")


--- Santa Catarina Report (1996) ---
Total of entries: 1764
Null entries (NaN): 1764 (100.00%)
Fluvial entries: 0


With that we can confirm, Santa Catarina didn't register any data that year, 100% of their values were registered as NaN

In [24]:
# Running the audit
quality_report = audit_missing_values(df_tourism)
display(quality_report)

,Year,Missing %,Total NaN,Affected States,Likely Causes
0,1989,3.45,588,[mato grosso do sul],State 'MATO GROSSO DO SUL' missing methods: ['fluvial']
1,1996,9.68,1764,[santa catarina],State 'SANTA CATARINA' registered NO data (100% missing)
2,1999,3.23,588,[distrito federal],State 'DISTRITO FEDERAL' registered NO data (100% missing)
3,2004,9.26,1920,"[amazonas, bahia, ceará, pará, paraná, pernambuco, rio grande do norte, rio grande do sul, rio de janeiro, santa catarina, são paulo, outras unidades da federação, mato grosso do sul]","State 'AMAZONAS' missing methods: ['aérea', 'terrestre'] | State 'BAHIA' missing methods: ['aérea', 'marítima'] | State 'CEARÁ' missing methods: ['aérea', 'marítima'] | State 'PARÁ' missing methods: ['aérea', 'fluvial'] | State 'PARANÁ' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'PERNAMBUCO' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO NORTE' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO SUL' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'RIO DE JANEIRO' missing methods: ['aérea', 'marítima'] | State 'SANTA CATARINA' missing methods: ['aérea', 'marítima', 'terrestre'] | State 'SÃO PAULO' missing methods: ['aérea', 'marítima'] | State 'OUTRAS UNIDADES DA FEDERAÇÃO' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'MATO GROSSO DO SUL' missing methods: ['terrestre']"
4,2007,10.71,2304,"[amazonas, bahia, ceará, pará, paraná, pernambuco, rio grande do norte, rio grande do sul, rio de janeiro, santa catarina, são paulo, outras unidades da federação, mato grosso do sul]","State 'AMAZONAS' missing methods: ['aérea', 'terrestre'] | State 'BAHIA' missing methods: ['aérea', 'marítima'] | State 'CEARÁ' missing methods: ['aérea', 'marítima'] | State 'PARÁ' missing methods: ['aérea', 'fluvial'] | State 'PARANÁ' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'PERNAMBUCO' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO NORTE' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO SUL' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'RIO DE JANEIRO' missing methods: ['aérea', 'marítima'] | State 'SANTA CATARINA' missing methods: ['aérea', 'marítima', 'terrestre'] | State 'SÃO PAULO' missing methods: ['aérea', 'marítima'] | State 'OUTRAS UNIDADES DA FEDERAÇÃO' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'MATO GROSSO DO SUL' missing methods: ['terrestre']"
5,2012,8.82,2016,"[paraná, rio grande do sul]","State 'PARANÁ' missing methods: ['marítima', 'fluvial'] | State 'RIO GRANDE DO SUL' missing methods: ['marítima']"
6,2014,2.44,672,[mato grosso do sul],State 'MATO GROSSO DO SUL' missing methods: ['fluvial']
